# 06. Convolutional Neural Network Architectures 

<div style="margin:.3rem 0 1rem;font-size:.9em;color:#555;display:flex;align-items:center;gap:.35rem;font-family:monospace">
  <time datetime="2026-03-05">05 Mar 2026</time> 
</div>

<a href="https://colab.research.google.com/github/shahaliyev/csci4701/blob/main/docs/notebooks/06_cnn_architectures.ipynb"
   target="_blank" rel="noopener">
  <img
    src="https://colab.research.google.com/assets/colab-badge.svg"
    alt="Open in Colab"
  />
</a>

<div class="admonition warning">
  <p class="admonition-title">Important</p>
  <p style="margin: 1em 0;">
    The notebook is currently under revision.
  </p>
</div>

<div class="admonition info">
  <p class="admonition-title">Info</p>
  <p style="margin: 0.5em 0;">
    The following source was consulted in preparing this material: Zhang, A., Lipton, Z. C., Li, M., &amp; Smola, A. J. <a href="https://d2l.ai/">Dive into Deep Learning</a>. Cambridge University Press. <a href="https://d2l.ai/chapter_convolutional-modern/index.html">Chapter 8: Modern Convolutional Neural Networks</a>.
  </p>
</div>

For about a decade convolutional architectures dominated computer vision. Around 2020, however, a major shift occurred with the introduction of [Vision Transformers (ViT)](https://en.wikipedia.org/wiki/Vision_transformer). Follow-up work demonstrated that attention-based models could match or exceed the performance of many convolutional networks when trained on large datasets, which decreased the popularity of convolutional architectures. Despite this, architectures like ConvNeXt ([Liu et al., 2022](https://arxiv.org/abs/2201.03545)) revisited the CNN design<span class="fn"><span class="fn-body">These included using larger depthwise convolution kernels, replacing ReLU with GELU activations, simplifying the residual block structure, adjusting the placement of normalization layers, and adopting training practices that had become common in transformer models. </span></span> and showed that much of the performance gap came from differences in training practices rather than architectural limits. With these changes, convolutional networks were able to achieve results close to transformer-based models on ImageNet. 

Today both approaches are widely used. Transformers are common in large-scale vision systems and multimodal models, while convolutional networks remain important due to their efficiency and simplicity, especially in applications where computation and memory are limited. Moreover, many ideas used in modern architectures, such as hierarchical feature extraction, residual connections, etc. were first developed in convolutional networks.


In previous notebooks we introduced basic convolutional neural networks such as [LeNet](../03_cnn_torch) and discussed optimization challenges like [vanishing gradients](../04_regul_optim). In this notebook we will discuss influential neural network architectures and their core ideas.

## ImageNet

We discussed in our introduction that the progress in [deep learning](../../introduction/01_deep_learning) was limited by the lack of large labeled datasets. A major turning point came with the introduction of the *Imagenet* ([Deng et al., 2009](https://www.image-net.org/static_files/papers/imagenet_cvpr09.pdf)) dataset. The dataset was organized according to the [WordNet hierarchy](https://wordnet.princeton.edu/), a large lexical database of English nouns. The idea was to collect thousands of real-world images illustrating a wide range of classes so that machine learning systems could learn visual concepts at scale. Images were gathered from search engines and then verified by humans (crowdsourced) threefold using [Amazon Mechanical Turk](https://www.mturk.com/). The full ImageNet dataset contains more than _14 million images_ at $224 \times 224$ resolution, covering over _20,000 categories_, which took about three years and (perhaps) a couple of hundred thousand dollars to build. 

<figure>
  <img src="../../assets/images/overview/alexnet.png" alt="AI/ML/DL relation" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
   Eight ILSVRC-2010 test images and the five labels considered most probable by the AlexNet model. Performance in the challenge is typically measured using <em>top-1 accuracy</em>, which checks whether the model's most confident prediction is correct, and <em>top-5 accuracy</em>, which checks whether the correct class appears among the five most confident predictions. The top-5 metric was originally introduced because many classes in ImageNet are visually similar and fine-grained. ~ Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). <a href='https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf'>ImageNet Classification with Deep Convolutional Neural Networks</a>. Advances in Neural Information Processing Systems (NeurIPS)  
  </figcaption>
</figure>

Most deep learning research focuses on a subset of introduced for the [ImageNet Large Scale Visual Recognition Challenge (ILSVRC)](https://www.image-net.org/challenges/LSVRC/index.php) benchmark, which was launched to evaluate algorithms for image classification and object detection. The classification task uses *1000 object categories*, each with roughly *1000 training images*. This results in approximately *1.2 million* training images, along with a validation set of about *50,000* images and a separate test set used for leaderboard evaluation.

## AlexNet

AlexNet ([Krizhevsky et al., 2012](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks)) won ILSVRC 2012 by a large margin. Architecturally it was very close to LeNet: a sequence of convolutional and pooling layers followed by fully connected layers for classification. The main reasons it succeeded where earlier networks had not were basically much more data, better hardware, and a few critical design and training choices.

<figure>
  <img src="../../assets/images/cnn_architectures/alexnet.svg" alt="AlexNet architecture" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    LeNet and AlexNet architectures ~ Zhang et al., <a href="https://d2l.ai/chapter_convolutional-modern/alexnet.html">Dive into Deep Learning</a>, <a href="https://d2l.ai/_images/alexnet.svg">Fig. 8.1.2</a>.
    <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  </figcaption>
</figure>

The increase in data and resolution made it possible (and necessary) to use a larger, deeper network in AlexNet. LeNet used [sigmoid](../../mathematics/03_probability#sigmoid) (and tanh) activations, which **saturate** for large positive or negative inputs, so gradients shrink and learning slows. AlexNet switched to ReLU, which does not saturate for $x > 0$, so gradients flow more easily and training of deeper nets became feasible. This change alone had a large impact on convergence speed and final accuracy.

Training AlexNet on a single GPU would have been extremely slow and difficult due to memory constraints. The GPUs available at the time (GTX 580 with 3GB memory) could not comfortably store the entire network and its intermediate activations, especially because the model contained very large fully connected layers. To address this, the authors split the network across two GPUs. Each GPU processed a subset of feature maps in many layers, and the two streams communicated only at specific points in the network. 

<div class="admonition note">
  <p class="admonition-title">Note</p>
  <p style="margin: 1em 0;">
    The <a href='https://www.cs.toronto.edu/~kriz/'>codebase written in C++</a> by <a href='https://en.wikipedia.org/wiki/Alex_Krizhevsky'>Alexei Krizhevsky</a> was used for a few following years to train major CNN models. Today, deep learning frameworks handle GPU parallelism automatically. We usually define a single model and the framework distributes computation across devices using techniques such as data parallelism. Manually splitting a network across GPUs, as done in AlexNet, is rarely necessary today.
  </p>
</div>
 
AlexNet made heavy use of [dropout](../04_regul_optim#dropout) in the fully connected layers to reduce overfitting, and [data augmentation](04_regul_optim#data-augmentation) to effectively enlarge the training set. 

AlexNet also replaced the average pooling with max pooling. Max pooling outputs the largest activation in the window, preserving the strongest detected feature rather than averaging responses together. This helps the network retain the most salient signals during downsampling. AlexNet also introduced overlapping pooling, using $3 \times 3$ pooling windows with stride $2$. Because the windows overlap, adjacent pooling regions share some pixels. The authors reported that this design slightly reduced overfitting and improved classification performance.


<div class="admonition note">
  <p class="admonition-title">Note</p>
  <p style="margin: 1em 0;">
    Some ideas introduced in the paper did not remain widely used. For example, <em>local response normalization (LRN)</em> has largely been replaced by batch normalization, which stabilizes training more effectively.
  </p>
</div>

## VGGNet

VGG ([Simonyan and Zisserman, 2015](https://arxiv.org/abs/1409.1556)) replaced the mix of kernel sizes in AlexNet with a simple, repetitive design: _modular blocks_. A VGG block is a small stack of $3 \times 3$ conv–ReLU layers (often two or three) then one $2 \times 2$ max pooling layer. The whole network is a sequence of such blocks, with channel counts increasing (e.g. 64, 128, 256, 512). This uniform structure made it easy to scale depth (VGG11, VGG16, VGG19) and influenced later designs.

<div class="admonition note">
  <p class="admonition-title">Note</p>
  <p style="margin: 1em 0;">
    Simply stacking more VGG blocks eventually led to training difficulties (beyond the depth of 19 VGG returned diminishing returns), which residual connections and batch normalization later addressed.
  </p>
</div>

<figure>
  <img src="../../assets/images/cnn_architectures/vgg.svg" alt="VGGNet" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    Alexnet and VGG architectures ~ Zhang et al., <a href="https://d2l.ai/chapter_convolutional-modern/vgg.html">Dive into Deep Learning</a>, <a href="https://d2l.ai/_images/vgg.svg">Fig. 8.2.1</a>.
    <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  </figcaption>
</figure>

To understand the architecture, it is important to explain the one more concept.  The **receptive field** of a neuron is the set of input pixels that can influence its value. A single $5 \times 5$ convolution gives each output pixel a receptive field of $5 \times 5$: it sees 25 input pixels.  

<figure>
  <img src="../../assets/images/cnn_architectures/receptive5.png" alt="Receptive field of 5x5 kernel" style="max-width: 75%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    The receptive field of a single 5x5 kernel (plotted offline with matplotlib)
  </figcaption>
</figure>

It turns out, two $3 \times 3$ convolutions in sequence also yield a receptive field of $5 \times 5$: the first $3 \times 3$ sees 9 pixels, and the second $3 \times 3$ sees 9 *output* values of the first layer, which themselves each see 9 input pixels. So one $5 \times 5$ and two $3 \times 3$ layers have the same receptive field.

<figure>
  <img src="../../assets/images/cnn_architectures/receptive3.png" alt="Receptive field of 3x3 kernel" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    The receptive field of two 3x3 kernels (plotted offline with matplotlib)
  </figcaption>
</figure>

With the same receptive field, two $3 \times 3$ layers are strictly more expressive: they insert an extra nonlinearity between the two convolutions, so the function is a composition of two linear plus nonlinear steps instead of one. Hence, the **representational power** of two kernels is higher. 

At the same time they use fewer parameters. For $C$ input and output channels, a single $5 \times 5$ layer has $25 C^2$ parameters. Two $3 \times 3$ layers have $9 C^2 + 9 C^2 = 18 C^2$ parameters — about 30% fewer — while covering the same spatial context and allowing more flexible, hierarchical features. The same reasoning extends to three $3 \times 3$ layers replacing one $7 \times 7$: same receptive field, more non-linearities, fewer parameters.

## Network in Network

Network in Network (NiN) ([Lin et al., 2014](https://arxiv.org/abs/1312.4400)) introduced two major ideas that became standard.

<figure>
  <img src="../../assets/images/cnn_architectures/nin.svg" alt="Network in Network" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    VGGNet and Network in Network ~ Zhang et al., <a href="https://d2l.ai/chapter_convolutional-modern/nin.html">Dive into Deep Learning</a>, <a href="https://d2l.ai/_images/nin.svg">Fig. 8.3.1</a>.
    <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  </figcaption>
</figure>


At first glance, a kernel of size $1 \times 1$ seems meaningless, as it does not combine information across height or width — only across channels. However, at each spatial position $(h, w)$, the layer takes the $C_{\text{in}}$ channel values and computes $C_{\text{out}}$ linear combinations of them. So it behaves like a small fully connected layer applied independently at every pixel. That gives the network a way to mix and compress channels (e.g. reduce 256 channels to 64) or add extra non-linear capacity without changing spatial resolution.

<figure>
  <img src="../../assets/images/cnn_architectures/nin_1x1.svg" alt="The effect of 1x1 kernel" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    The effect of 1x1 kernel (plotted offline with matplotlib)
  </figcaption>
</figure>

After the last convolutional layers we usually have a tensor of shape (batch, channels, height, width). Classical designs flatten this and feed it into one or more large fully connected layers to produce class scores. Those dense layers contain most of the parameters and are prone to overfitting. **Global average pooling** layer introduced by NiN replaces that with a single step: _average each channel over all spatial positions_. So for each of the $C$ channels we compute one number (the mean over $H \times W$). The result is a vector of length $C$, which can be fed directly into the classifier (or a single linear layer). No flattening, no huge matrices to keep in the memory, just one number per channel. That drastically cuts parameters, reduces overfitting, and makes the output invariant to the spatial size of the feature map, so the same network can handle slightly different input resolutions. Since NiN, GAP is common in modern CNNs.

<figure>
  <img src="../../assets/images/cnn_architectures/gap.svg" alt="Global average pooling (GAP) layer" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    The effect of the global average pooling layer on a simple input (plotted offline with matplotlib)
  </figcaption>
</figure>

## GoogLeNet and Inception Modules

<div class="admonition info">
  <p class="admonition-title">Info</p>
  <p style="margin: 0.5em 0;">
    Based on Zhang, A., Lipton, Z. C., Li, M., &amp; Smola, A. J. <a href="https://d2l.ai/">Dive into Deep Learning</a>, <a href="https://d2l.ai/chapter_convolutional-modern/googlenet.html">Section 8.4: Networks with Parallel Concatenations (GoogLeNet)</a>.
  </p>
</div>

GoogLeNet ([Szegedy et al., 2014](https://arxiv.org/abs/1409.4842)), also called Inception v1, introduced the **Inception block**: instead of choosing one kernel size per layer, the block runs **several convolutions in parallel** (e.g. $1 \times 1$, $3 \times 3$, $5 \times 5$) and **concatenates** their outputs along the channel dimension. The network can thus combine fine-grained ($1 \times 1$), local ($3 \times 3$), and broader ($5 \times 5$) patterns in one place, and the relative importance of each path is learned.

A naive version would be very expensive: many channels through $5 \times 5$ convolutions. GoogLeNet therefore uses **$1 \times 1$ convolutions as bottlenecks** before the larger kernels: they reduce the channel count, so the subsequent $3 \times 3$ and $5 \times 5$ layers do far fewer operations while still capturing multi-scale structure. The full Inception module typically has four branches (e.g. $1 \times 1$ only; $1 \times 1$ then $3 \times 3$; $1 \times 1$ then $5 \times 5$; $3 \times 3$ max-pool then $1 \times 1$), with the $1 \times 1$ layers keeping cost down. GoogLeNet also uses global average pooling and **auxiliary classifiers** (small heads attached to intermediate layers during training) to improve gradient flow and act as regularizers.

<figure>
  <img src="../../assets/images/cnn_architectures/inception-module.svg" alt="Inception module" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    Inception module with parallel paths ~ Zhang et al., <a href="https://d2l.ai/chapter_convolutional-modern/googlenet.html">Dive into Deep Learning</a>, <a href="https://d2l.ai/_images/inception.svg">Fig. 8.4.2</a>.
    <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  </figcaption>
</figure>

<figure>
  <img src="../../assets/images/cnn_architectures/googlenet.svg" alt="GoogLeNet architecture" style="max-width: 100%; height: auto;">
  <figcaption style="margin-top: 0.5em; font-size: 0.9em; opacity: 0.85;">
    GoogLeNet architecture composed of Inception modules ~ Zhang et al., <a href="https://d2l.ai/chapter_convolutional-modern/googlenet.html">Dive into Deep Learning</a>, <a href="https://d2l.ai/_images/inception-full.svg">Fig. 8.4.3</a>.
    <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>.
  </figcaption>
</figure>

<div class="admonition note">
  <p class="admonition-title">Note</p>
  <p style="margin: 1em 0;">
  The name <em>GoogLeNet</em> is a play on words: Google and <a href="../03_cnn_torch">LeNet</a>.
  </p>
</div>


## DenseNet

[DenseNet](https://arxiv.org/abs/1608.06993) ([Huang et al., 2017](https://arxiv.org/abs/1608.06993)) takes the idea of shortcut connections further than [ResNet](./05_batchnorm_resnet). In a **dense block**, each layer receives the concatenation of the feature maps from **all previous layers** in that block, not just the immediate predecessor. So the $l$-th layer has inputs from layers $0, 1, \ldots, l-1$. This maximizes information flow: every layer can reuse all preceding features. To keep the number of channels manageable, each layer typically produces only a small number of new feature maps (e.g. 12 or 32), and **transition layers** (e.g. $1 \times 1$ conv plus pooling) between dense blocks reduce spatial size and sometimes channel count. DenseNet achieves strong accuracy with fewer parameters than many comparable ResNets, at the cost of higher memory use during training because of the concatenations.

## SqueezeNet

[SqueezeNet](https://arxiv.org/abs/1602.07360) ([Iandola et al., 2016](https://arxiv.org/abs/1602.07360)) aims for **very small models** (e.g. under 1MB) without giving up too much accuracy. The main building block is the **fire module**: a **squeeze** layer made of $1 \times 1$ convolutions (reducing channels) followed by two parallel paths—**expand** layers of $1 \times 1$ and $3 \times 3$ convolutions—whose outputs are concatenated. By using many $1 \times 1$ filters and few $3 \times 3$ filters, SqueezeNet keeps the parameter count low. It is useful when model size or inference cost is constrained (e.g. embedded or mobile).

## MobileNet

[MobileNet](https://arxiv.org/abs/1704.04861) ([Howard et al., 2017](https://arxiv.org/abs/1704.04861)) is designed for **efficient inference on mobile and edge devices**. The core idea is **depthwise separable convolution**: factor a standard convolution into (1) a **depthwise** convolution—one $3 \times 3$ filter per channel, no cross-channel mixing—and (2) a **pointwise** $1 \times 1$ convolution that mixes channels. This drastically reduces parameters and FLOPs compared to a single $3 \times 3$ conv with the same input/output channels, with relatively small accuracy loss. MobileNet v2 adds **inverted residual blocks** and linear bottlenecks to improve accuracy and efficiency further. These designs are widely used when latency or power matters.

## EfficientNet

[EfficientNet](https://arxiv.org/abs/1905.11946) ([Tan and Le, 2019](https://arxiv.org/abs/1905.11946)) addresses the question of how to scale a baseline CNN (depth, width, resolution) in a balanced way. Instead of scaling one dimension arbitrarily, the authors use a **compound scaling** method: scale depth, width, and input resolution together via a set of exponents derived from a small grid search. The base architecture is built from **mobile inverted bottleneck** (MBConv) blocks—similar to MobileNet v2—and is then scaled to produce a family of models (EfficientNet-B0 through B7) that achieve strong accuracy versus FLOPs and parameter count. EfficientNet is often used when one needs a range of accuracy–efficiency trade-offs with a single, consistent design.

## Accessing Architectures in PyTorch

Modern deep learning frameworks provide reference implementations of these architectures. In PyTorch, <code>torchvision.models</code> includes AlexNet, VGG variants, GoogLeNet, DenseNet, MobileNet, EfficientNet, SqueezeNet, and many others. You can inspect their layer structure and use pretrained weights (e.g. <code>weights='IMAGENET1K_V1'</code>) for transfer learning or benchmarking.


In [ ]:
from torchvision import models

# Classic architectures
alexnet = models.alexnet(weights=None)
vgg16 = models.vgg16(weights=None)
googlenet = models.googlenet(weights=None)

# Efficient / mobile and dense variants
densenet121 = models.densenet121(weights=None)
mobilenet_v3_small = models.mobilenet_v3_small(weights=None)
efficientnet_b0 = models.efficientnet_b0(weights=None)
squeezenet1_0 = models.squeezenet1_0(weights=None)

# Inspect a few (e.g. layer counts, parameter counts)
alexnet.features[:6], vgg16.features[:10], googlenet.conv1


<div class="admonition success">
  <p class="admonition-title">Exercise</p>
  <p style="margin: 1em 0;">
    Compare the total number of parameters in AlexNet, VGG16, GoogLeNet, DenseNet-121, MobileNet v3 Small, EfficientNet-B0, and SqueezeNet using <code>sum(p.numel() for p in model.parameters())</code>. Relate the differences to design choices: stacking small convolutions (VGG), $1 \times 1$ bottlenecks and GAP (GoogLeNet, NiN), dense connections (DenseNet), depthwise separable convolutions (MobileNet), compound scaling (EfficientNet), and fire modules (SqueezeNet).
  </p>
</div>
